# DigiStructMed — Resume from Step 3a

**Use this notebook when you already have Step 1 outputs** (`resolved_entities.json`, `text_blocks.json`, `table_index.json`, tables CSVs).

Steps 1a–1e are skipped entirely. Step 2 (ontology loading) runs in-memory because Steps 3a/3b need the `ontology` object.

| Step | Name | Status |
|------|------|--------|
| 1a–1e | Source structuring | Skipped — upload outputs below |
| 2  | Ontology Loading | Runs (in-memory, fast) |
| 3a | Table → RML Mappings | Runs |
| 3b | Text Path (v1 \| v2) | Runs |
| 4  | KG Materialization (SDM-RDFizer / Morph-KGC) | Runs |
| 5  | SHACL Validation (TravSHACL / pySHACL fallback) | Runs |

---
## 0. Setup

In [ ]:
!pip install -q pymupdf pdfplumber rdflib transformers accelerate sentencepiece
!pip install -q rdfizer
!pip install -q morph-kgc
!pip install -q travshacl
!pip install -q pyshacl
!pip install -q docling
!pip install -q huggingface_hub
!pip install -q rapidfuzz

In [ ]:
import os, sys, zipfile
from pathlib import Path
from typing import Optional

UPLOAD_PROJECT_ZIP = True


def _find_project_with_scripts(here: Path) -> Optional[Path]:
    if (here / 'scripts').is_dir():
        return here.resolve()
    for child in sorted(here.iterdir()):
        if child.is_dir() and (child / 'scripts').is_dir():
            return child.resolve()
    return None


root = Path.cwd().resolve()

if UPLOAD_PROJECT_ZIP:
    try:
        from google.colab import files
        print(
            'Click **Choose Files** below, then upload DigiStructMed_thesis_colab.zip '
            '(from create_colab_zip.py on your PC).'
        )
        uploaded = files.upload()
        for name, data in uploaded.items():
            dest = root / name
            dest.write_bytes(data)
            print(f'Received {len(data) // 1024} KB -> {name}')
            if name.lower().endswith('.zip'):
                with zipfile.ZipFile(dest, 'r') as zf:
                    zf.extractall(root)
                print('ZIP extracted.')
            break
    except ImportError:
        print('Not running in Colab — using current folder (no upload).')

proj = _find_project_with_scripts(root)
if proj is not None and proj != root:
    os.chdir(proj)
    root = Path.cwd().resolve()
    print(f'Using project folder: {root}')

if not (root / 'scripts').is_dir():
    raise RuntimeError(
        'Cannot find scripts/ — upload the thesis Colab ZIP above, or set '
        'UPLOAD_PROJECT_ZIP = False and %cd into the repo root first.'
    )

scripts_dir = str(root / 'scripts')
if scripts_dir not in sys.path:
    sys.path.insert(0, scripts_dir)

print('Working directory:', root)
print('scripts/ on path:', scripts_dir)

---
## 0b. Hugging Face token

Needed for the LLM fallback in Step 3a, Step 3b v2, and any gated model download.

In [ ]:
from getpass import getpass

print(
    'Hugging Face token (hf_...): needed for gated model download and HF Inference API.\n'
    'Leave empty if you will not use HF (3a LLM fallback / 3b v2 will skip LLM).'
)
_hf = getpass('HF_TOKEN: ')
HF_TOKEN = _hf.strip() or None
if HF_TOKEN:
    print('HF_TOKEN set (length', len(HF_TOKEN), 'chars).')
else:
    print('HF_TOKEN not set.')

In [ ]:
ONTOLOGY_PATH = 'input/hf_guideline_ontology.ttl'

HF_MODEL = 'meta-llama/Llama-3.1-8B-Instruct'

if HF_TOKEN:
    LLM_BACKEND = 'hf_local'
else:
    LLM_BACKEND = 'none'

TEXT_PATH_VERSION = 'v2'   # 'v1' | 'v2'

print(f'Ontology        : {ONTOLOGY_PATH}')
print(f'LLM backend     : {LLM_BACKEND}  (HF_MODEL={HF_MODEL})')
print(f'Text path ver   : {TEXT_PATH_VERSION}')

---
## 1. Upload existing Step 1 outputs

This cell detects which Step 1 output files are already on disk and prompts you to upload any that are missing.

| File | Required by |
|------|-------------|
| `resolved_entities.json` | Steps 3b + 4 |
| `text_blocks.json` | Step 3b |
| `table_index.json` | Step 3a |
| `tables/*.csv` | Step 3a (the actual table data) |

In [ ]:
from pathlib import Path

try:
    from google.colab import files as _colab_files
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

STEP1_DIR = Path('outputs/step1')
STEP1_DIR.mkdir(parents=True, exist_ok=True)
(STEP1_DIR / 'tables').mkdir(parents=True, exist_ok=True)

_REQUIRED = {
    'resolved_entities.json': 'Steps 3b + 4 — entity->CUI mapping',
    'text_blocks.json':        'Step 3b — text assertions source',
}
_OPTIONAL = {
    'table_index.json': 'Step 3a — table mappings (skip if you have no tables)',
}

for fname, desc in _REQUIRED.items():
    p = STEP1_DIR / fname
    if p.is_file():
        print(f'OK  {fname}  ({p.stat().st_size // 1024} KB)  — {desc}')
    else:
        print(f'X   {fname}  NOT FOUND  — {desc}')
        if _IN_COLAB:
            print(f'   -> Upload {fname} now:')
            up = _colab_files.upload()
            for _name, _data in up.items():
                p.write_bytes(_data)
                print(f'   Saved -> {p}  ({len(_data) // 1024} KB)')
                break
        else:
            print(f'   -> Copy the file to: {p.resolve()}')

for fname, desc in _OPTIONAL.items():
    p = STEP1_DIR / fname
    if p.is_file():
        print(f'OK  {fname}  ({p.stat().st_size // 1024} KB)  — {desc}')
    else:
        print(f'!   {fname}  not found  — {desc}')
        if _IN_COLAB:
            print(f'   -> Upload {fname} for the table path, or skip (cancel) for text-only KG:')
            up = _colab_files.upload()
            for _name, _data in up.items():
                p.write_bytes(_data)
                print(f'   Saved -> {p}  ({len(_data) // 1024} KB)')
                break
        else:
            print(f'   -> Copy to: {p.resolve()}  (or skip for text-only KG)')

csv_files = list((STEP1_DIR / 'tables').glob('*.csv'))
if csv_files:
    print(f'OK  tables/  — {len(csv_files)} CSV files found')
else:
    print('!   tables/  — no CSVs found. Upload table CSVs into outputs/step1/tables/ for Step 3a.')
    if _IN_COLAB:
        print('   -> Upload table CSV files now (select all at once):')
        up = _colab_files.upload()
        for _name, _data in up.items():
            dest = STEP1_DIR / 'tables' / _name
            dest.write_bytes(_data)
            print(f'   Saved -> {dest}')

print()
_missing_req = [f for f in _REQUIRED if not (STEP1_DIR / f).is_file()]
if not _missing_req:
    print('All required Step 1 outputs present — proceed to Step 2.')
else:
    print('Missing required files:', _missing_req)

---
## Upload ontology (if not already in the ZIP)

In [ ]:
from pathlib import Path

ont_p = Path(ONTOLOGY_PATH)
if ont_p.is_file():
    print(f'OK  Ontology already present: {ont_p}  ({ont_p.stat().st_size // 1024} KB)')
else:
    print(f'X   Ontology not found at {ont_p}')
    try:
        from google.colab import files as _cf
        print('   -> Upload your ontology .ttl file:')
        up = _cf.upload()
        for _name, _data in up.items():
            ont_p.parent.mkdir(parents=True, exist_ok=True)
            ont_p.write_bytes(_data)
            print(f'   Saved -> {ont_p}  ({len(_data) // 1024} KB)')
            break
    except ImportError:
        print(f'   -> Copy the file to: {ont_p.resolve()}')

---
## Force-reload all pipeline modules

Colab caches imported modules in memory. If you re-uploaded the ZIP mid-session, the old code stays loaded. This cell forces Python to re-read every script from disk.

In [ ]:
import importlib, sys

_pipeline_modules = [
    'hf_llm',
    'step2_load_ontology',
    'step3a_table_mappings',
    'step3b_text_path',
    'step4_materialize',
    'step5_validate',
]

for _mod_name in _pipeline_modules:
    if _mod_name in sys.modules:
        importlib.reload(sys.modules[_mod_name])
        print(f'  reloaded {_mod_name}')
    else:
        print(f'  {_mod_name} — not yet imported (will load fresh)')

print('\nAll pipeline modules will use the latest code from disk.')

---
## Step 2 — Ontology Loading

Loads the ontology into memory. Required by Steps 3a and 3b.

In [ ]:
from step2_load_ontology import load_ontology

ontology = load_ontology(ONTOLOGY_PATH)

print(f'Ontology summary:')
print(f'  Classes           : {len(ontology.classes)}')
print(f'  Object properties : {len(ontology.object_properties)}')
print(f'  Datatype properties: {len(ontology.datatype_properties)}')
print(f'  Named individuals : {len(ontology.named_individuals)}')
print(f'  Enumerations      : {len(ontology.enumerations)}')

---
## Step 3a — Table -> RML Mappings

In [ ]:
from step3a_table_mappings import generate_table_mappings

table_mapping_result = generate_table_mappings(
    table_index_path='outputs/step1/table_index.json',
    ontology=ontology,
    output_dir='outputs/step3',
    llm_backend=LLM_BACKEND,
    hf_token=HF_TOKEN,
    hf_model=HF_MODEL,
    use_llm_fallback=True,
)
print('\nTable mapping counts:', table_mapping_result['counts'])
print('Review file (check before Step 4):', table_mapping_result['review_path'])

---
## Step 3b — Text Path

In [ ]:
from step3b_text_path import run_text_path

text_result = run_text_path(
    resolved_entities_path='outputs/step1/resolved_entities.json',
    text_blocks_path='outputs/step1/text_blocks.json',
    ontology=ontology,
    output_dir='outputs/step3',
    version=TEXT_PATH_VERSION,
    llm_backend=LLM_BACKEND,
    hf_token=HF_TOKEN,
    hf_model=HF_MODEL,
)
print(f"\n-> [{TEXT_PATH_VERSION}] {text_result['count']} text assertions")

---
## Step 4 — KG Materialization

In [ ]:
from step4_materialize import materialize

kg_path = materialize(
    table_mappings_path='outputs/step3/table_mappings.ttl',
    text_mappings_path=f'outputs/step3/text_mappings_{TEXT_PATH_VERSION}.ttl',
    text_assertions_path=f'outputs/step3/text_assertions_{TEXT_PATH_VERSION}.json',
    resolved_entities_path='outputs/step1/resolved_entities.json',
    ontology_path=ONTOLOGY_PATH,
    output_dir='outputs/step4',
    version=TEXT_PATH_VERSION,
)
print(f'\n-> KG written to: {kg_path}')

---
## Step 5 — SHACL Validation

In [ ]:
from step5_validate import validate

SPARQL_ENDPOINT = None

report = validate(
    kg_path=f'outputs/step4/output_{TEXT_PATH_VERSION}.ttl',
    ontology_path=ONTOLOGY_PATH,
    output_dir='outputs/step5',
    version=TEXT_PATH_VERSION,
    sparql_endpoint=SPARQL_ENDPOINT,
)

print(f"\nBackend         : {report['backend']}")
print(f"Conforms        : {report['conforms']}")
print(f"Total triples   : {report['total_triples']}")
print(f"Violations      : {report['total_violations']}")
print(f"Conformance     : {report['metrics']['conformance_ratio']:.1%}")

---
## Run Both Versions & Compare

Runs v1 and v2 of the text path, materializes both, validates both, and shows a side-by-side comparison.

In [ ]:
from pathlib import Path
from step3b_text_path import run_text_path
from step4_materialize import materialize
from step5_validate   import validate, compare_versions

for ver in ('v1', 'v2'):
    sep = '=' * 50
    print(f'\n{sep}')
    print(f'  Running version: {ver}')
    print(sep)

    assertions_file = Path(f'outputs/step3/text_assertions_{ver}.json')
    if assertions_file.is_file():
        print(f'  -> Step 3b [{ver}] already exists ({assertions_file.stat().st_size // 1024} KB) — reusing')
    else:
        run_text_path(
            resolved_entities_path='outputs/step1/resolved_entities.json',
            text_blocks_path='outputs/step1/text_blocks.json',
            ontology=ontology,
            output_dir='outputs/step3',
            version=ver,
            llm_backend=LLM_BACKEND,
            hf_token=HF_TOKEN,
            hf_model=HF_MODEL,
        )

    kg_file = Path(f'outputs/step4/output_{ver}.ttl')
    if kg_file.is_file():
        print(f'  -> Step 4 [{ver}] already exists ({kg_file.stat().st_size // 1024} KB) — reusing')
    else:
        materialize(
            table_mappings_path='outputs/step3/table_mappings.ttl',
            text_mappings_path=f'outputs/step3/text_mappings_{ver}.ttl',
            text_assertions_path=f'outputs/step3/text_assertions_{ver}.json',
            resolved_entities_path='outputs/step1/resolved_entities.json',
            ontology_path=ONTOLOGY_PATH,
            output_dir='outputs/step4',
            version=ver,
        )

    validate(
        kg_path=f'outputs/step4/output_{ver}.ttl',
        ontology_path=ONTOLOGY_PATH,
        output_dir='outputs/step5',
        version=ver,
    )

compare_versions()

---
## Download Results

In [ ]:
import shutil
from pathlib import Path

shutil.make_archive('DigiStructMed_outputs', 'zip', 'outputs')

try:
    from google.colab import files
    files.download('DigiStructMed_outputs.zip')
except ImportError:
    print('Not running in Colab — outputs ZIP is at: DigiStructMed_outputs.zip')